# Introdução e questão de pesquisa geral


Este notebook analisa se há associações entre alertas SonarQube e alucinações anotadas nas respostas de código geradas por LLMs. 

Os dados usados serão gerado de:

- `../input.jsonl`: respostas geradas por LLMs, avaliações e anotações humanas de alucinação.
- `../mapped_issues.json`: alertas SonarQube mapeados para as respostas geradas.

Cada linha de `input.jsonl` representa uma resposta gerada; as issues do SonarQube são vinculadas a essa resposta por `row_number`.


## Questão de pesquisa

> Questão 1: A presença de alertas SonarQube está associada a categorias específicas de alucinação em código gerado por LLMs? 

> Questão 2: Essa associação se mantém quando se analisa por categoria específica de alerta (`BUG` vs. `CODE_SMELL`)?

Essa questão será respondida em duas etapas, correspondentes às duas seções principais deste notebook:

1. **Presença de alerta (agregada) × tipo de alucinação**: testamos se a simples presença de 
   qualquer alerta do SonarQube (independente do tipo) está associada a cada categoria de 
   alucinação anotada.
2. **Tipo de alerta × tipo de alucinação** : testamos se tipos específicos 
   de alerta (`BUG`, `CODE_SMELL`) têm associação diferenciada com cada categoria de alucinação, 
   permitindo identificar padrões que a primeira análise poderia mascarar.

Em ambas as etapas, o objetivo não é apenas identificar **se** existe associação estatisticamente 
significativa, mas também quantificar a **força e a direção** dessa associação, e ser transparente 
sobre as limitações que uma análise desse tipo carrega.

As discussões dos resultados obtidos serão discutidas na seção final.

## Metodologia

O processo de análise segue estas etapas:

1. **Unidade de análise**: cada resposta de código gerada por um LLM é tratada como um caso 
   independente, identificado por `row_number`.

2. **Construção da tabela 2×2**: para cada combinação entre uma variável de alerta SonarQube 
   (presença geral, ou tipo específico) e uma categoria de alucinação, construímos uma tabela 
   de contingência 2×2.

3. **Cálculo das frequências esperadas**: a partir da tabela observada, calculamos as frequências 
   esperadas sob a hipótese de independência entre as duas variáveis.

4. **Escolha do teste estatístico**: aplicamos o critério de Cochran sobre as frequências esperadas 
   para decidir entre o teste qui-quadrado de Pearson (frequências adequadas) ou o teste exato 
   de Fisher (frequências pequenas/esparsas).

5. **Cálculo do p-valor**: obtemos o p-valor do teste escolhido, que indica a evidência estatística 
   contra a hipótese de independência entre as variáveis.

6. **Ajuste por múltiplas comparações**: como várias combinações são testadas simultaneamente, 
   ajustamos os p-valores brutos usando Bonferroni e FDR (Benjamini-Hochberg), para controlar 
   o risco de falsos positivos.

7. **Cálculo do tamanho de efeito**: para cada combinação, calculamos o odds ratio (com correção 
   de Haldane-Anscombe) e seu intervalo de confiança de 95%, para quantificar a força e a direção 
   da associação.

8. **Critério de conclusão**: consideramos uma associação estatisticamente robusta apenas quando 
   o p-valor ajustado é menor que 0,05, calculando para essas associações a magnitude 
   do efeito (odds ratio) e sua precisão (intervalo de confiança).

Cada uma dessas etapas é detalhada nas subseções a seguir.

### Construção da tabela 2×2

Para cada hipótese, cruzamos a presença de um tipo de alucinação com a presença de alerta SonarQube na mesma resposta:

Para a hipótese de presença de alerta por tipo de alucinação:

| | Alerta presente | Alerta ausente |
|---|---:|---:|
| Tipo de alucinação presente | a | b: |
| Tipo de alucinação ausente | c | d|


Para a hipótese de categoria de alerta por tipo de alucinação:

| | Categoria presente | Categoria ausente |
|---|---:|---:|
| Tipo de alucinação presente | a | b |
| Tipo de alucinação ausente | c | d |


- a ambos presentes
- b só alucinação presente
- c só alerta/categoria presente |
- d ambos ausentes

### Frequências esperadas

Frequência esperada é o número de casos que deveríamos observar em cada célula de uma tabela de contingência se as duas variáveis fossem completamente independentes, ou seja, se não houvesse associação nenhuma entre elas.

A frequência esperada para cada célula é:

$$E_{ij} = \frac{(\text{total da linha}_i)(\text{total da coluna}_j)}{N}$$

O resultado da frequência esperada será usada no teste de Coshran e para o cálculo do qui-quadrado.


### Regra de Cochran e escolha do teste

Para tabelas  2×2, a regra de Cochran diz que no máximo 20% das células podem ter frequência esperada <5 para que o teste qui-quadrado seja considerado confiávelAssim, 
- Se que todas as quatro frequências esperadas devem ser $≥ 5$ aplicaremos a aproximação do qui-quadrado.
- Se alguma frequência esperada for menor que 5, escolhemos o teste exato de Fisher. 

Nesta tabela de quatro células, esse teste coincide com a regra de no máximo $20%$ das células abaixo de 5: 20% de 4 é 0,8 célula. A implementação do qui-quadrado não aplica a correção de continuidade de Yates.

### Qui-quadrado de Pearson

O qui-quadrado de Pearson mede o quanto os valores observados numa tabela de contingência se afastam dos valores esperados sob a hipótese de independência — e converte esse "afastamento total" numa estatística que, comparada a uma distribuição de referência, gera o p-valor.

Assume-se que as duas variáveis são independentes (não há associação). Sob essa suposição, calculamos as frequências esperadas em cada célula, como vimos na resposta anterior.

Para cada célula da tabela, calculamos a diferença entre o que foi observado e o que era esperado: $Desvio= Observado - Esperado$

Se a hipótese nula fosse perfeitamente verdadeira, esse desvio seria zero em todas as células.

Quando elevamos o Desvio ao quadrado, garantimos que a "distância" entre o real e o esperado do será sempre positiva e, penaliza grandes desvios.

Para normalizar os desvios, os dividimos pelo valor esperado.

Contudo, 

$$\chi^2=\sum \frac{(Observado - Esperado)^2}{Esperado}$$

indica o quanto, no total, a tabela observada se afasta da tabela esperada sob independência.

**p-valor** é a área sob a curva da distribuição qui-quadrado (com k graus de liberdade), começando no ponto onde está o χ² observado e indo até o infinito. Quanto maior o χ², mais para a direita esse ponto de corte fica, e menor a área (p-valor) restante depois dele.

A distribuição qui-quadrado é uma aproximação contínua de algo que, na origem, é discreto. Essa aproximação só funciona bem quando as frequências esperadas são grandes o suficiente para que a distribuição dos possíveis valores de χ² se pareça de fato com uma curva qui-quadrado suave. Quando alguma célula tem frequência esperada muito baixa (< 5), a aproximação fica ruim, e a curva contínua deixa de representar bem essa realidade. É exatamente por isso que, nesses casos, o Fisher se torna a alternativa mais confiável.

O resultado do qui-quadrado aparece nas colunas:

- `chi2`
- `chi2_p_value`




### Teste exato de Fisher

O **teste exato de Fisher** avalia se há associação entre duas variáveis categóricas em uma tabela 2x2.

Ele foi usado porque algumas categorias possuem poucos exemplos, como `Common Sense`, `Inconsistent Libraries` e `Useless Statements (unexecuted)`. Nesses casos, o qui-quadrado pode ser menos confiável, pois depende de aproximação assintótica.

Hipóteses:

- **H0:** não há associação entre o tipo de alucinação e a presença de alerta SonarQube.
- **H1:** há associação entre o tipo de alucinação e a presença de alerta SonarQube.

Dado que os totais das linhas e colunas da tabela já são conhecidos, qual é a probabilidade de observar exatamente essa distribuição nas 4 células internas, e qual seria a probabilidade de todas as outras distribuições possíveis com os mesmos totais?

Por exemplo, se eu simplesmente distribuísse 369 'alertas' aleatoriamente entre as 1134 respostas, qual a chance de, por puro acaso, exatamente 31 desses alertas caírem justamente nas 60 respostas com Algorithm?
Se 

- a ambos presentes
- b só alucinação presente
- c só alerta/categoria presente |
- d ambos ausentes
- $C(x,y)$ o número de formas de selecionar y itens de um total de x, sem se importar com a ordem

então a probabilidade é dada por:

$$P(a) = \frac{C(a+b, a) × C(c+d, c)}{C(N, a+c)}$$

Calcular P(a) para a tabela observada é só o primeiro passo. O p-valor de Fisher soma a probabilidade da tabela observada mais a probabilidade de todas as outras tabelas possíveis (com os mesmos totais marginais) que sejam tão extremas ou mais extremas que a observada, na direção testada:

**p-valor** e a soma da probabilidade de todas as tabelas que têm probabilidade menor ou igual que a probabilidade da tabela observada.

O p-valor de Fisher aparece na coluna `fisher_exact_p_value`.





### Correção para múltiplas comparações

Para limitar falsos positivos ao testar várias hipóteses, ajustamos os p-valores recomendados por Bonferroni e Benjamini–Hochberg (FDR), separadamente por seção:

- **A:** $m$ é o número de tipos de alucinação testados;
- **B:** $m$ é o número de pares categoria SonarQube × tipo de alucinação.

Bonferroni multiplica o p-valor por $m$, limitado a 1. Benjamini–Hochberg controla a taxa esperada de falsas descobertas entre os resultados significativos. Consideramos o resultado significativo após ajuste quando $p_{ajustado}<0,05$.


### Odds ratio (OR)

O odds ratio quantifica a direção e o tamanho da associação:

$$OR = \frac{ad}{bc}$$

- **OR > 1**: chance maior de alerta/categoria quando a alucinação está presente.
- **OR < 1**: chance menor de alerta/categoria quando a alucinação está presente.
- **OR = 1**: nenhuma associação.

Quando alguma célula é zero, o OR bruto fica indefinido (divisão por zero) ou nulo (numerador zero). 
Para evitar esse problema, aplicamos a **correção de Haldane–Anscombe** (soma de 0,5 em cada célula):

$$OR_c = \frac{(a+0{,}5)(d+0{,}5)}{(b+0{,}5)(c+0{,}5)}$$



### Intervalo de confiança do OR

O IC 95% é calculado pelo método de Wald, na escala logarítmica do OR corrigido:

$$SE = \sqrt{\frac{1}{a+0{,}5} + \frac{1}{b+0{,}5} + \frac{1}{c+0{,}5} + \frac{1}{d+0{,}5}}$$

$$IC_{95\%} = \exp\left(\log(OR_c) \pm 1{,}96 \times SE\right)$$

> **Nota:** esse intervalo é calculado individualmente para cada tabela e **não** é ajustado 
> para múltiplas comparações — a correção por múltiplos testes (Bonferroni/FDR) é aplicada 
> separadamente sobre os p-valores, não sobre o OR ou seu intervalo.


### Correção por múltiplas comparações: Bonferroni e FDR (Benjamini-Hochberg)

Como cada categoria de alucinação é testada separadamente contra os alertas do SonarQube, 
realizamos $m$ testes de hipótese simultâneos (um por categoria). Isso aumenta o risco de 
encontrar associações "significativas" apenas por acaso: mesmo que nenhuma associação real 
exista, a probabilidade de obter pelo menos um falso positivo entre os $m$ testes é:

$$P(\text{pelo menos 1 falso positivo}) = 1 - (1 - \alpha)^m$$

Com $\alpha = 0{,}05$ e $m$ testes, essa probabilidade cresce rapidamente conforme $m$ aumenta, 
tornando necessário ajustar o critério de significância.

A correção **FDR/Benjamini-Hochberg** controla a taxa esperada de falsas descobertas entre os resultados declarados significativos. Como a análise possui caráter exploratório e envolve múltiplas categorias relacionadas, adotamos Benjamini–Hochberg (FDR = 0,05) como correção principal.


#### FDR / Benjamini-Hochberg

Controla a **taxa de falsas descobertas** (*false discovery rate*, FDR) — a proporção 
esperada de falsos positivos **entre os resultados considerados significativos**, em vez do 
risco de qualquer falso positivo ocorrer. O procedimento ordena os $m$ p-valores brutos do 
menor para o maior ($p_{(1)} \leq p_{(2)} \leq ... \leq p_{(m)}$) e compara cada um a um 
limiar que cresce conforme a posição no ranking:

$$p_{(i)} \leq \frac{i}{m} \times \alpha$$

Isso torna o método **menos conservador** que Bonferroni, mantendo mais achados como 
significativos, ao custo de tolerar uma proporção maior de falsos positivos entre eles.

Nesse notebook, esse dado aparece na coluna `fdr_bh_p_value` e consideramos significância em `alpha = 0.05`.


#### Critério adotado

Para cada tabela 2×2, aplicamos as duas correções ao **p-valor recomendado** (qui-quadrado 
ou Fisher, conforme o critério de Cochran). O ajuste é feito **separadamente para cada seção 
de análise** (presença de alerta × alucinação; tipo de alerta × alucinação), já que cada 
seção representa uma família de testes distinta, com seu próprio valor de $m$.

Consideramos uma associação estatisticamente significativa apenas quando o **p-valor 
ajustado** (Bonferroni e/ou FDR, conforme reportado) é menor que 0,05.



## Arquivos utilizados

Esta análise parte dos arquivos derivados anteriormente a partir de `input.jsonl` e `mapped_issues.json`.

Arquivos principais:

- `response_level_dataset.csv`: uma linha por resposta gerada, contendo os tipos de alucinação, impactos anotados e indicação de presença/ausência de alerta SonarQube.
- `sonar_presence_by_hallucination_type.csv`: tabela 2x2 por tipo de alucinação, com testes estatísticos e correções de múltiplas comparações.
- `sonar_presence_by_affection.csv`: tabela 2x2 por impacto anotado, com testes estatísticos e correções de múltiplas comparações.
- `issue_level_dataset.csv`: uma linha por alerta SonarQube, útil para análises descritivas no nível de issue.
- `analysis_summary.json`: resumo da quantidade de respostas, issues e rótulos.

O foco inferencial principal está nos arquivos `sonar_presence_by_hallucination_type.csv` e `sonar_presence_by_affection.csv`.

In [52]:
from pathlib import Path
import json
import math
from collections import Counter, defaultdict
from IPython.display import Markdown, display

import pandas as pd

START_DIR = Path.cwd().resolve()
DATA_DIR = START_DIR.parent if START_DIR.name == "statistical_analysis" else START_DIR

INPUT_JSONL = DATA_DIR / "metadata.jsonl"
MAPPED_ISSUES = DATA_DIR / "mapped_issues.json"

print("Pasta dos dados:", DATA_DIR)
print("metadata.jsonl:", INPUT_JSONL)
print("mapped_issues.json:", MAPPED_ISSUES)

In [53]:
input_rows = []
with INPUT_JSONL.open(encoding="utf-8") as f:
    for row_number, line in enumerate(f, start=1):
        obj = json.loads(line)
        input_rows.append((row_number, obj))

with MAPPED_ISSUES.open(encoding="utf-8") as f:
    mapped_issues = json.load(f)

issues_by_row = defaultdict(list)
for issue in mapped_issues:
    issues_by_row[int(issue["row_number"])].append(issue)

print("Respostas em metadata.jsonl:", len(input_rows))
print("Issues SonarQube mapeadas:", len(mapped_issues))
print("Respostas com pelo menos uma issue:", len(issues_by_row))

Respostas em metadata.jsonl: 3120
Issues SonarQube mapeadas: 1426
Respostas com pelo menos uma issue: 838


In [ ]:
records = []
for row_number, obj in input_rows:
    labels = obj.get("code_labels") or []
    hallucination_types = sorted({
        label.get("hallucination-type")
        for label in labels
        if label.get("hallucination-type")
    })
    affections = sorted({
        affection
        for label in labels
        for affection in (label.get("affections") or [])
        if affection
    })
    issues = issues_by_row.get(row_number, [])
    records.append({
        "row_number": row_number,
        "_id": obj.get("_id"),
        "model": obj.get("model"),
        "dataset": obj.get("dataset"),
        "level": obj.get("level"),
        "project": obj.get("project"),
        "has_sonar_issue": bool(issues),
        "sonar_issue_count": len(issues),
        "sonar_rules": sorted({issue.get("sonar_rule") for issue in issues if issue.get("sonar_rule")}),
        "sonar_types": sorted({issue.get("sonar_type") for issue in issues if issue.get("sonar_type")}),
        "sonar_severities": sorted({issue.get("severity") for issue in issues if issue.get("severity")}),
        "hallucination_types": hallucination_types,
        "affections": affections,
    })

df = pd.DataFrame(records)
df.head()

,row_number,_id,model,dataset,level,project,has_sonar_issue,sonar_issue_count,sonar_rules,sonar_types,sonar_severities,hallucination_types,affections
0,1,636767191a6d9265ec017c0f,deepseek-coder-1.3b,CEJava,None,hasor-master,True,2,[java:S1854],[CODE_SMELL],[MAJOR],[],[]
1,2,636767191a6d9265ec017c0f,deepseek-coder-7b,CEJava,None,hasor-master,False,0,[],[],[],[],[]
2,3,636767191a6d9265ec017c0f,codellama-7b,CEJava,None,hasor-master,True,2,[java:S1854],[CODE_SMELL],[MAJOR],[],[]
3,4,636767191a6d9265ec017c0f,gpt-4,CEJava,None,hasor-master,True,1,[java:S1144],[CODE_SMELL],[MAJOR],[],[]
4,5,636767191a6d9265ec017c0f,deepseek-r1,CEJava,None,hasor-master,True,2,[java:S1854],[CODE_SMELL],[MAJOR],[Library/Project],[Incorrect Functionality]


In [ ]:
df["sonar_rules"].explode().dropna().unique()

<StringArray>
[  'java:S1854',   'java:S1144',   'java:S1168',   'java:S1125',
   'java:S1488',    'java:S106',   'java:S1126',    'java:S117',
   'java:S3776',   'java:S1172',   'java:S1066',   'java:S1197',
   'java:S1604',   'java:S1075',   'java:S3012',   'java:S1871',
    'java:S127',   'java:S2293',   'java:S1149',    'java:S112',
   'java:S1135',   'java:S1481',   'java:S6201',   'java:S1192',
   'java:S3626',   'java:S1130',   'java:S6208',   'java:S1659',
   'java:S3358',   'java:S1206',    'java:S135',   'java:S3740',
   'java:S2129',   'java:S5738',    'java:S125',   'java:S4719',
   'java:S2189', 'python:S1172', 'python:S6326', 'python:S3776',
 'python:S5603', 'python:S2275', 'python:S5754',  'python:S112',
 'python:S1481', 'python:S1135', 'python:S1854',  'python:S117',
 'python:S1226', 'python:S3457', 'python:S1862', 'python:S5864',
 'python:S1656', 'python:S2190', 'python:S5756', 'python:S1192',
 'python:S5361', 'python:S6019', 'python:S6395',  'python:S930',
 'python:S1

A primeira verificação é confirmar o tamanho da amostra e a quantidade de respostas com e sem alerta SonarQube.

Neste desenho, `has_sonar_issue = true` significa que pelo menos uma issue do SonarQube foi mapeada para aquela resposta gerada. Uma resposta pode ter várias issues, mas para os testes principais ela conta apenas uma vez.

In [ ]:
summary = {
    "Total de Respostas": len(df),
    "Respostas com Problema no Sonar": int(df["has_sonar_issue"].sum()),
    "Respostas sem Problema no Sonar": int((~df["has_sonar_issue"]).sum()),
    "Total de Problemas no Sonar": len(mapped_issues),
}
summary

{'Total de Respostas': 3120,
 'Respostas com Problema no Sonar': 838,
 'Respostas sem Problema no Sonar': 2282,
 'Total de Problemas no Sonar': 1426}

## Funções Utilizadas

In [57]:
def expected_counts(a, b, c, d):
    total = a + b + c + d
    row_with_hallucination = a + b
    row_without_hallucination = c + d
    column_with_sonar = a + c
    column_without_sonar = b + d
    return [
        row_with_hallucination * column_with_sonar / total,
        row_with_hallucination * column_without_sonar / total,
        row_without_hallucination * column_with_sonar / total,
        row_without_hallucination * column_without_sonar / total,
    ]

In [58]:
def cochran_pass_2x2(a, b, c, d):
    return min(expected_counts(a, b, c, d)) >= 5

In [59]:
def chi_square_2x2(a, b, c, d):
    total = a + b + c + d
    denominator = (a + b) * (c + d) * (a + c) * (b + d)
    if denominator == 0:
        return math.nan, math.nan
    statistic = total * (a * d - b * c) ** 2 / denominator
    p_value = math.erfc(math.sqrt(statistic / 2))
    return statistic, p_value

In [60]:
def hypergeom_probability(observed_a, row_total, column_total, total):
    return (
        math.comb(column_total, observed_a)
        * math.comb(total - column_total, row_total - observed_a)
        / math.comb(total, row_total)
    )

In [61]:
def fisher_exact_2x2(a, b, c, d):
    row_total = a + b
    column_total = a + c
    total = a + b + c + d
    minimum_a = max(0, row_total - (total - column_total))
    maximum_a = min(row_total, column_total)
    observed_probability = hypergeom_probability(a, row_total, column_total, total)
    p_value = 0.0
    for candidate_a in range(minimum_a, maximum_a + 1):
        probability = hypergeom_probability(
            candidate_a, row_total, column_total, total
        )
        if probability <= observed_probability + 1e-12:
            p_value += probability
    return min(p_value, 1.0)

In [62]:
def odds_ratio_corrected(a, b, c, d):
    if 0 in (a, b, c, d):
        a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5

    return (a * d) / (b * c)

In [63]:
def odds_ratio_wald_ci_95(a, b, c, d):
    corrected_cells = [a + 0.5, b + 0.5, c + 0.5, d + 0.5]
    corrected_a, corrected_b, corrected_c, corrected_d = corrected_cells
    log_odds_ratio = math.log(
        (corrected_a * corrected_d) / (corrected_b * corrected_c)
    )
    standard_error = math.sqrt(sum(1 / cell for cell in corrected_cells))
    z_critical = 1.959963984540054
    return (
        math.exp(log_odds_ratio - z_critical * standard_error),
        math.exp(log_odds_ratio + z_critical * standard_error),
    )

In [64]:
def phi_coefficient(a, b, c, d):
    denominator = (a + b) * (c + d) * (a + c) * (b + d)
    if denominator == 0:
        return math.nan
    return (a * d - b * c) / math.sqrt(denominator)

In [65]:
def benjamini_hochberg(p_values):
    number_of_tests = len(p_values)
    adjusted = [None] * number_of_tests
    previous = 1.0
    sorted_indexes = sorted(
        range(number_of_tests),
        key=lambda index: p_values[index],
        reverse=True,
    )
    for rank_from_largest, index in enumerate(sorted_indexes):
        rank = number_of_tests - rank_from_largest
        raw_adjusted = p_values[index] * number_of_tests / rank
        previous = min(previous, raw_adjusted)
        adjusted[index] = min(previous, 1.0)
    return adjusted

In [66]:
def build_pairwise_association_results(pairs):
    rows = []
    for pair in pairs:
        has_sonar = pair["has_sonar"]
        has_hallucination = pair["has_hallucination"]
        a = int((has_sonar & has_hallucination).sum())
        b = int((~has_sonar & has_hallucination).sum())
        c = int((has_sonar & ~has_hallucination).sum())
        d = int((~has_sonar & ~has_hallucination).sum())
        expected = expected_counts(a, b, c, d)
        use_chi_square = cochran_pass_2x2(a, b, c, d)
        chi2, chi2_p_value = chi_square_2x2(a, b, c, d)
        fisher_p_value = fisher_exact_2x2(a, b, c, d)
        recommended_p_value = chi2_p_value if use_chi_square else fisher_p_value
        rate_with_hallucination = a / (a + b) if a + b else math.nan
        rate_without_hallucination = c / (c + d) if c + d else math.nan
        odds_ratio = odds_ratio_corrected(a, b, c, d)
        ci_lower, ci_upper = odds_ratio_wald_ci_95(a, b, c, d)
        rows.append({
            "sonar_type": pair["sonar_type"],
            "hallucination_type": pair["hallucination_type"],
            "a_both_present": a,
            "b_hallucination_only": b,
            "c_sonar_only": c,
            "d_neither_present": d,
            "responses_with_hallucination": a + b,
            "responses_with_sonar": a + c,
            "sonar_rate_with_hallucination": rate_with_hallucination,
            "sonar_rate_without_hallucination": rate_without_hallucination,
            "risk_difference": rate_with_hallucination - rate_without_hallucination,
            "expected_counts": expected,
            "expected_min": min(expected),
            "cochran_pass": use_chi_square,
            "recommended_test": "chi-square" if use_chi_square else "fisher-exact",
            "chi2": chi2,
            "chi2_p_value": chi2_p_value,
            "fisher_exact_p_value": fisher_p_value,
            "recommended_p_value": recommended_p_value,
            "odds_ratio_corrected": odds_ratio,
            "odds_ratio_ci95_lower": ci_lower,
            "odds_ratio_ci95_upper": ci_upper,
            "or_ci95_excludes_1": ci_lower > 1 or ci_upper < 1,
            "phi": phi_coefficient(a, b, c, d),
        })

    results = pd.DataFrame(rows)
    if results.empty:
        return results

    number_of_tests = len(results)
    results["fdr_bh_p_value"] = benjamini_hochberg(
        results["recommended_p_value"].tolist()
    )
    results["significant_fdr_bh_0_05"] = results["fdr_bh_p_value"] < 0.05
    return results.sort_values(
        ["fdr_bh_p_value", "recommended_p_value"]
    ).reset_index(drop=True)

In [67]:
def render_association_interpretation(results, analysis_label):
    lines = [f"### Interpretação: {analysis_label}", ""]
    for _, row in results.iterrows():
        if row["significant_fdr_bh_0_05"]:
            evidence = "associação significativa após FDR-BH, mas não após Bonferroni"
        else:
            evidence = "sem associação significativa após os ajustes"

        odds_ratio = row["odds_ratio_corrected"]
        if odds_ratio > 1:
            direction = "a taxa de alertas é maior quando a categoria de alucinação está presente"
        elif odds_ratio < 1:
            direction = "a taxa de alertas é menor quando a categoria de alucinação está presente"
        else:
            direction = "as odds estimadas são iguais nos dois grupos"

        lines.append(
            f"- **{row['sonar_type']} × {row['hallucination_type']}:** {evidence}; "
            f"taxas {row['sonar_rate_with_hallucination']:.1%} vs. "
            f"{row['sonar_rate_without_hallucination']:.1%}; OR = {odds_ratio:.2f}, "
            f"IC95% [{row['odds_ratio_ci95_lower']:.2f}, "
            f"{row['odds_ratio_ci95_upper']:.2f}]; {direction}. "
            f"Phi = {row['phi']:.3f}."
        )

    return "\n".join(lines)

## Presença de alerta × tipo de alucinação


Para cada tipo de alucinação, a pergunta estatística é:

> Respostas com esse tipo de alucinação têm uma frequência diferente de alertas SonarQube quando comparadas às respostas sem esse tipo?



A tabela abaixo apresenta as quatro contagens da 2×2 e a taxa de respostas com alerta, comparando respostas com e sem cada tipo de alucinação.

In [ ]:
all_hallucination_types = sorted(
    {value for values in df["hallucination_types"] for value in values}
 )

pairs_a = []
any_sonar_alert = df["has_sonar_issue"].astype(bool)
for hallucination_type in all_hallucination_types:
    has_hallucination = df["hallucination_types"].apply(
        lambda values: hallucination_type in values
    )
    pairs_a.append({
        "sonar_type": "Qualquer alerta SonarQube",
        "hallucination_type": hallucination_type,
        "has_sonar": any_sonar_alert,
        "has_hallucination": has_hallucination,
    })

hallucination_results = build_pairwise_association_results(pairs_a)
print(f"Número de categorias testadas (m): {len(hallucination_results)}")
table_rate_columns_a = [
    "hallucination_type",
    "a_both_present",
    "b_hallucination_only",
    "c_sonar_only",
    "d_neither_present",
    "responses_with_hallucination",
    "sonar_rate_with_hallucination",
    "sonar_rate_without_hallucination",
    "risk_difference",
]
display(hallucination_results[table_rate_columns_a])

Número de categorias testadas (m): 12


,hallucination_type,a_both_present,b_hallucination_only,c_sonar_only,d_neither_present,responses_with_hallucination,sonar_rate_with_hallucination,sonar_rate_without_hallucination,risk_difference
0,Library/Project,120,189,718,2093,309,0.388350,0.255425,0.132924
1,Algorithm,32,28,806,2254,60,0.533333,0.263399,0.269935
2,Undefined Variables,82,118,756,2164,200,0.410000,0.258904,0.151096
3,Useless Statements (executed without effect),35,36,803,2246,71,0.492958,0.263365,0.229593
4,Data Conflicting,6,44,832,2238,50,0.120000,0.271010,-0.151010
5,Fragmented Logics,2,19,836,2263,21,0.095238,0.269764,-0.174526
6,Mathematics & Natural Science,2,15,836,2267,17,0.117647,0.269417,-0.151770
7,Useless Statements (unexecuted),3,4,835,2278,7,0.428571,0.268230,0.160341
8,Behavior Conflicting,116,304,722,1978,420,0.276190,0.267407,0.008783
9,Computer Theory,8,19,830,2263,27,0.296296,0.268348,0.027948



`recommended_test` é escolhido a partir das frequências esperadas. `recommended_p_value` é o p-valor bruto usado na correção; os ajustes são feitos sobre todos os tipos de alucinação testados, com $m$ igual ao número de categorias.

In [69]:
test_columns_a = [
    "hallucination_type",
    "expected_min",
    "recommended_test",
    "chi2_p_value",
    "fisher_exact_p_value",
    "recommended_p_value",
    "fdr_bh_p_value",
    "significant_fdr_bh_0_05",
]
display(hallucination_results[test_columns_a])

,hallucination_type,expected_min,recommended_test,chi2_p_value,fisher_exact_p_value,recommended_p_value,fdr_bh_p_value,significant_fdr_bh_0_05
0,Library/Project,82.994231,chi-square,5.617111e-07,0.000001,5.617111e-07,0.000007,True
1,Algorithm,16.115385,chi-square,2.984436e-06,0.000012,2.984436e-06,0.000012,True
2,Undefined Variables,53.717949,chi-square,3.101537e-06,0.000007,3.101537e-06,0.000012,True
3,Useless Statements (executed without effect),19.069872,chi-square,1.597275e-05,0.000058,1.597275e-05,0.000048,True
4,Data Conflicting,13.429487,chi-square,1.685877e-02,0.015299,1.685877e-02,0.040461,True
5,Fragmented Logics,5.640385,chi-square,7.211849e-02,0.084422,7.211849e-02,0.144237,False
6,Mathematics & Natural Science,4.566026,fisher-exact,1.591358e-01,0.269527,2.695266e-01,0.462046,False
7,Useless Statements (unexecuted),1.880128,fisher-exact,3.390457e-01,0.394641,3.946410e-01,0.591961,False
8,Behavior Conflicting,112.807692,chi-square,7.055867e-01,0.722611,7.055867e-01,0.893099,False
9,Computer Theory,7.251923,chi-square,7.442489e-01,0.827378,7.442489e-01,0.893099,False


### Odds ratio e Indice de confiança de 95%

O intervalo de confiança é do OR com correção de Haldane–Anscombe. Verifique se inclui 1; ele é individual e não substitui os p-valores ajustados por Bonferroni/FDR.

In [70]:
effect_columns_a = [
    "hallucination_type",
    "odds_ratio_corrected",
    "odds_ratio_ci95_lower",
    "odds_ratio_ci95_upper",
    "or_ci95_excludes_1"
]
display(hallucination_results[effect_columns_a])

,hallucination_type,odds_ratio_corrected,odds_ratio_ci95_lower,odds_ratio_ci95_upper,or_ci95_excludes_1
0,Library/Project,1.850820,1.452237,2.363799,True
1,Algorithm,3.196030,1.915444,5.305169,True
2,Undefined Variables,1.989149,1.486099,2.670059,True
3,Useless Statements (executed without effect),2.719316,1.701291,4.346437,True
4,Data Conflicting,0.366805,0.171806,0.897872,True
5,Fragmented Logics,0.284941,0.092761,1.297396,False
6,Mathematics & Natural Science,0.361563,0.114691,1.666674,False
7,Useless Statements (unexecuted),2.046108,0.523520,8.593749,False
8,Behavior Conflicting,1.045378,0.832527,1.318486,False
9,Computer Theory,1.148003,0.528777,2.669178,False


### Resultados por tipo de alucinação

In [71]:
display(
    Markdown(
        render_association_interpretation(
            hallucination_results,
            "Seção A: qualquer alerta × tipo de alucinação",
        )
    )
)

### Interpretação: Seção A: qualquer alerta × tipo de alucinação

- **Qualquer alerta SonarQube × Library/Project:** associação significativa após FDR-BH, mas não após Bonferroni; taxas 38.8% vs. 25.5%; OR = 1.85, IC95% [1.45, 2.36]; a taxa de alertas é maior quando a categoria de alucinação está presente. Phi = 0.090.
- **Qualquer alerta SonarQube × Algorithm:** associação significativa após FDR-BH, mas não após Bonferroni; taxas 53.3% vs. 26.3%; OR = 3.20, IC95% [1.92, 5.31]; a taxa de alertas é maior quando a categoria de alucinação está presente. Phi = 0.084.
- **Qualquer alerta SonarQube × Undefined Variables:** associação significativa após FDR-BH, mas não após Bonferroni; taxas 41.0% vs. 25.9%; OR = 1.99, IC95% [1.49, 2.67]; a taxa de alertas é maior quando a categoria de alucinação está presente. Phi = 0.083.
- **Qualquer alerta SonarQube × Useless Statements (executed without effect):** associação significativa após FDR-BH, mas não após Bonferroni; taxas 49.3% vs. 26.3%; OR = 2.72, IC95% [1.70, 4.35]; a taxa de alertas é maior quando a categoria de alucinação está presente. Phi = 0.077.
- **Qualquer alerta SonarQube × Data Conflicting:** associação significativa após FDR-BH, mas não após Bonferroni; taxas 12.0% vs. 27.1%; OR = 0.37, IC95% [0.17, 0.90]; a taxa de alertas é menor quando a categoria de alucinação está presente. Phi = -0.043.
- **Qualquer alerta SonarQube × Fragmented Logics:** sem associação significativa após os ajustes; taxas 9.5% vs. 27.0%; OR = 0.28, IC95% [0.09, 1.30]; a taxa de alertas é menor quando a categoria de alucinação está presente. Phi = -0.032.
- **Qualquer alerta SonarQube × Mathematics & Natural Science:** sem associação significativa após os ajustes; taxas 11.8% vs. 26.9%; OR = 0.36, IC95% [0.11, 1.67]; a taxa de alertas é menor quando a categoria de alucinação está presente. Phi = -0.025.
- **Qualquer alerta SonarQube × Useless Statements (unexecuted):** sem associação significativa após os ajustes; taxas 42.9% vs. 26.8%; OR = 2.05, IC95% [0.52, 8.59]; a taxa de alertas é maior quando a categoria de alucinação está presente. Phi = 0.017.
- **Qualquer alerta SonarQube × Behavior Conflicting:** sem associação significativa após os ajustes; taxas 27.6% vs. 26.7%; OR = 1.05, IC95% [0.83, 1.32]; a taxa de alertas é maior quando a categoria de alucinação está presente. Phi = 0.007.
- **Qualquer alerta SonarQube × Computer Theory:** sem associação significativa após os ajustes; taxas 29.6% vs. 26.8%; OR = 1.15, IC95% [0.53, 2.67]; a taxa de alertas é maior quando a categoria de alucinação está presente. Phi = 0.006.
- **Qualquer alerta SonarQube × Common Sense:** sem associação significativa após os ajustes; taxas 33.3% vs. 26.9%; OR = 1.36, IC95% [0.22, 12.39]; a taxa de alertas é maior quando a categoria de alucinação está presente. Phi = 0.005.
- **Qualquer alerta SonarQube × Inconsistent Libraries:** sem associação significativa após os ajustes; taxas 0.0% vs. 26.9%; OR = 0.54, IC95% [0.03, 11.34]; a taxa de alertas é menor quando a categoria de alucinação está presente. Phi = -0.015.

### Conclusão sobre tipos de alucinação

Após a correção para múltiplas comparações, seis tipos de alucinação apresentam associação 
estatisticamente significativa com a presença de alertas do SonarQube pelo **FDR/Benjamini-Hochberg**; 

| Tipo de alucinação | Direção | OR (IC 95%) |
|---|---|---|
| Undefined Variables * | Mais alertas | 1,74 (1,27 – 2,37) |
| Algorithm * | Mais alertas | 2,32 (1,38 – 3,90) |
| Useless Statements (executed without effect) | Mais alertas | 1,88 (1,16 – 3,04) |
| Behavior Conflicting * | Menos alertas | 0,65 (0,50 – 0,84) |
| Data Conflicting * | Menos alertas | 0,29 (0,13 – 0,67) |
| Fragmented Logics | Menos alertas | 0,26 (0,07 – 0,98) |

- Os ORs são calculados com correção de Haldane-Anscombe, e os ICs são individuais, não ajustados 
  por múltiplas comparações.

- `Fragmented Logics` tem intervalo de confiança com limite superior próximo de 1 (0,98) e só é significativo pelo FDR; 
  deve ser tratado com mais cautela.
  
- Os demais tipos (Common Sense, Mathematics & Natural Science, Library/Project, Useless Statements 
  (unexecuted), Computer Theory e Inconsistent Libraries) não apresentaram evidência suficiente de 
  associação. Em Common Sense e Inconsistent Libraries os intervalos de confiança são muito largos (por exemplo, 
  0,75 – 283,79), refletindo poucos casos; o OR pontual não é informativo nesses casos.

- Os dados indicam que o SonarQube, detecta melhor problemas ligados à estrutura do código (Algorithm, Undefined Variables, 
Useless Statements) do que inconsistências semânticas (Behavior/Data Conflicting), que dependem 
do que o código deveria fazer. Essa explicação não foi testada.

- Trata-se de associação, não de causalidade. As respostas de um mesmo problema ou 
modelo não são totalmente independentes, então os p-valores e ICs devem ser lidos com cautela.

### Conclusão sobre impactos

Após a correção para múltiplas comparações, três impactos permanecem estatisticamente
significativos (p ajustado < 0,05) tanto por **Bonferroni** quanto por **FDR/Benjamini-Hochberg**:

| Impacto | Taxa de alerta com o impacto | Taxa de alerta sem o impacto | OR (IC 95%) |  p ajustado (FDR) |
|---|---|---|---|---|
| `Low Efficiency` | 54,0% | 29,9% | 2,75 (1,89 – 4,01) | 2,5 × 10⁻⁷ |
| `Poor Readability` | 43,1% | 30,7% | 1,71 (1,22 – 2,39) | 2,1 × 10⁻³ |
| `Incorrect Functionality` | 30,6% | 53,1% | 0,39 (0,26 – 0,59) | 1,4 × 10⁻⁵ |

- `Low Efficiency` e `Poor Readability` têm `OR > 1`: respostas com esses impactos têm maior
  chance de apresentar alerta do SonarQube do que respostas sem eles. O efeito é maior em
  `Low Efficiency` do que em `Poor Readability`.
- `Incorrect Functionality` tem `OR < 1`: respostas com esse impacto apresentam menor chance de
  alerta do que respostas sem ele (diferença de 22 pontos percentuais).
- Os ORs foram calculados com correção de Haldane-Anscombe, e os ICs são individuais, não
  ajustados por múltiplas comparações.

- Esse padrão é compatível com o fato de o SonarQube ser uma
ferramenta de análise estática, mais sensível a problemas de eficiência e legibilidade do que a
violações semânticas do requisito, que dependem do que o código deveria fazer. (Essa explicação
não foi testada)

**Obs:** trata-se de associação, não de causalidade. As respostas de um mesmo problema ou
de um mesmo modelo não são totalmente independentes, então os p-valores e ICs devem ser lidos
com cautela. Além disso, `Incorrect Functionality` está presente em 1038 respostas (a grande
maioria da amostra), e o grupo sem esse impacto é pequeno; a taxa de 53,1% nesse grupo tem
mais incerteza e vale ser interpretada com cuidado.

In [ ]:
all_affections = sorted(
    {impact for impact_list in df["affections"] for impact in impact_list}
 )

impact_pairs = []
any_sonar_alert = df["has_sonar_issue"].astype(bool)
for impact in all_affections:
    has_impact = df["affections"].apply(
        lambda impact_list: impact in impact_list
    )
    impact_pairs.append({
        "sonar_type": "Qualquer alerta SonarQube",
        "hallucination_type": impact,
        "has_sonar": any_sonar_alert,
        "has_hallucination": has_impact,
    })

affection_results = build_pairwise_association_results(impact_pairs)
affection_results = affection_results.rename(
    columns={"hallucination_type": "impact"}
 )

focus_impacts = [
    "Low Efficiency",
    "Poor Readability",
    "Incorrect Functionality",
]
impact_columns = [
    "impact",
    "responses_with_hallucination",
    "sonar_rate_with_hallucination",
    "sonar_rate_without_hallucination",
    "risk_difference",
    "recommended_test",
    "recommended_p_value",
    "fdr_bh_p_value",
    "odds_ratio_corrected",
    "odds_ratio_ci95_lower",
    "odds_ratio_ci95_upper",
    "or_ci95_excludes_1",
]
impact_table = affection_results.loc[
    affection_results["impact"].isin(focus_impacts), impact_columns
].copy()
impact_table["direcao_or"] = impact_table["odds_ratio_corrected"].apply(
    lambda odds_ratio: "Mais alertas" if odds_ratio > 1 else "Menos alertas"
    if odds_ratio < 1 else "Sem diferença estimada"
 )

print(f"Total de impactos testados (m): {len(affection_results)}")
print("Tabela dos impactos solicitados:")
display(impact_table)

Total de impactos testados (m): 4
Tabela dos impactos solicitados:


,impact,responses_with_hallucination,sonar_rate_with_hallucination,sonar_rate_without_hallucination,risk_difference,recommended_test,recommended_p_value,fdr_bh_p_value,odds_ratio_corrected,odds_ratio_ci95_lower,odds_ratio_ci95_upper,or_ci95_excludes_1,direcao_or
0,Low Efficiency,124,0.572581,0.256008,0.316573,chi-square,6.499270e-15,2.599708e-14,3.893115,2.698611,5.584990,True,Mais alertas
1,Poor Readability,167,0.419162,0.260075,0.159087,chi-square,6.405103e-06,1.281021e-05,2.053130,1.497316,2.824031,True,Mais alertas
2,Incorrect Functionality,1038,0.314066,0.245917,0.068148,chi-square,5.196666e-05,6.928888e-05,1.404001,1.191106,1.655520,True,Mais alertas


### Conclusão 

A análise indica associação estatisticamente significativa (p ajustado < 0,05) entre a presença de 
alertas do SonarQube e alguns tipos de alucinação anotados manualmente.

Os resultados mais consistentes, que se mantêm após a correção de Bonferroni, são:

- Maior presença de alertas em respostas com `Undefined Variables` e `Algorithm`.
- Menor presença de alertas em respostas com `Behavior Conflicting` e `Data Conflicting`.
- Associação positiva entre alertas e os impactos `Low Efficiency` e `Poor Readability`.
- Associação negativa entre alertas e o impacto `Incorrect Functionality`.

O padrão é compatível com o fato de o SonarQube, como ferramenta de 
análise estática, ser mais sensível a problemas de estrutura, qualidade e uso de elementos do código 
do que a falhas funcionais e semânticas, que dependem do que o código deveria fazer. Essa 
explicação não foi testada neste trabalho.

Assim, os alertas do SonarQube podem servir como um sinal complementar de possíveis problemas em 
código gerado por LLMs, mas com poder limitado: os tamanhos de efeito são modestos, a direção da 
associação varia entre categorias, e o estudo não avalia capacidade de detecção. O SonarQube não 
substitui avaliação funcional, testes automatizados ou análise semântica da conformidade com o 
requisito.

**Obs:** trata-se de associação, não de causalidade. As respostas de um mesmo problema ou de 
um mesmo modelo não são totalmente independentes, então os p-valores e ICs devem ser lidos com cautela.

### Limitações

- **Dependência entre observações.** As respostas de um mesmo problema, ou de um mesmo modelo, 
  não são totalmente independentes. Os testes de qui-quadrado e Fisher assumem independência; 
  com dados agrupados, os p-valores tendem a ser menores, e os intervalos de confiança mais estreitos, do que a 
  incerteza real justificaria. Resultados com p ajustado próximo de 0,05 (como `Fragmented Logics`) 
  são os mais sensíveis a isso.

- Trata-se de um estudo observacional: os resultados indicam 
  associação, não causalidade. Fatores como modelo de LLM, dificuldade do problema e tamanho do 
  código podem explicar parte das associações observadas.

- Categorias com poucos casos (como `Common Sense` e `Inconsistent Libraries`) 
  produzem estimativas instáveis, com ICs muito largos; nesses casos, o OR pontual não é informativo.

- Uma resposta pode ter mais de um tipo de alucinação, e cada 
  categoria foi testada separadamente; portanto, os testes não são independentes entre si.

- Os tamanhos de efeito são modestos (phi entre 0,09 e 0,16 
  nos resultados significativos) e a direção varia entre categorias. O estudo avalia associação, 
  não a capacidade do SonarQube de detectar alucinações.

- Os intervalos de confiança são individuais e não foram ajustados por múltiplas 
  comparações.

### Trabalhos futuros

- Repetir a análise separadamente por modelo de LLM. Se a direção dos efeitos se mantiver nos 
  três, isso reforçaria a conclusão, mesmo com p-valores menos extremos devido ao N menor.
- Aplicar regressão logística com GEE (problema como grupo, modelo como covariável) para 
  estimar ORs ajustados e corrigir os erros-padrão pela correlação entre respostas do mesmo problema.

## Categoria de Alerta x tipo de alucinação